# Initial results of creating knowledge graphs for research papers

## Steps 1 & 2: Pip Installs And Imports

In [103]:
# Step 1: Do pip installs
packages = ["networkx", "pyvis", "PyMuPDF",
            "requests", "spacy", "regex",
            "keybert", "transformers", "sentence-transformers",
            "nltk", "scikit-learn",
            "python-dotenv", "google-genai"]

def install(package: str):
  !uv pip install -q {package}

for package in packages:
  install(package)

print("Installs finished!")

Installs finished!


In [104]:
# Step 2: All necessary imports
# NOTE: If you see "RuntimeError: CPU dispatcher tracer already initialized" after pip installs,
# restart the Jupyter kernel and re-run this cell.
import networkx as nx
from pyvis.network import Network
import fitz # PyMuPDF
import requests
import spacy
import itertools
from collections import Counter
import re
from keybert import KeyBERT

from transformers import pipeline
from itertools import combinations
from collections import defaultdict

from sentence_transformers import SentenceTransformer, util
import numpy as np

from IPython.display import HTML
import sklearn

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import PerceptronTagger
from nltk.corpus.reader.wordnet import NOUN, VERB, ADJ, ADV
import string

In [105]:
import os
import google.genai as genai
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configure the Gemini API key
gemini_api_key = os.getenv("GEMINI_API_KEY")
if gemini_api_key:
    client = genai.Client(api_key=gemini_api_key)
    print("Gemini API key configured.")
else:
    print("Gemini API key not found. Please set it in the .env file.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini API key configured.


In [106]:
# Download spaCy model only if not already installed
import spacy.util
model_name = 'en_core_web_sm'
if not spacy.util.is_package(model_name):
    # This will always get the latest version
    !uv pip install -q spacy
    !python -m spacy download en_core_web_sm

In [107]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\MSadm\AppData\Roaming\nltk

True

## Step 3: Text Extraction

In [108]:
def pdf_to_text(path: str) -> str:
    with fitz.open(path) as doc:
        return "\n".join(page.get_text() for page in doc)

def load_paper_text_from_url(paper_url: str, paper_filename: str) -> str:
  response = requests.get(paper_url)
  with open(paper_filename, "wb") as f:
    f.write(response.content)

  return pdf_to_text(paper_filename)

In [109]:
try:
    import google.colab
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Google Colab. Skipping file upload.")

def load_paper_text_from_file(paper_path: str):
  doc = fitz.open(paper_path)
  text = ""
  for page in doc:
      text += page.get_text()
  return text

Not running in Google Colab. Skipping file upload.


In [110]:
# paper_url = "https://zhenlab.com/wp-content/uploads/2025/02/Transfer_learning_paper___Bioinformatics_Advances.pdf"
# paper_filename = "zhenlab_paper.pdf"
# paper_text = load_paper_text_from_url(paper_url, paper_filename)

# paper_path = "Can MOOC Instructor Be Portrayed by Semantic Features.pdf"
# paper_text = load_paper_text_from_file(paper_path)
# print(f"Sample text from paper:\n{paper_text[:500]}...")

In [111]:
# load multiple papers and store them in a dictionary
def load_multiple_papers(paper_paths: list):
    """
    Load multiple papers and return a dictionary with paper path as key and text as value
    """
    papers = {}
    for paper_path in paper_paths:
        try:
            papers[paper_path] = load_paper_text_from_file(paper_path)
            print(f"Successfully loaded: {paper_path}")
        except Exception as e:
            print(f"Failed to load {paper_path}: {e}")
    
    return papers

## Step 4: Creation of Nodes

#### baseline nodes

In [112]:
# Step 4: Creates nodes
# Note this is a section of the code where there are multiple strategies
# to do this step. Some examples include:
#     - Named-Entity Recognition
#     - Extract noun phrases
#     - Extract ranked phrases with KeyBERT

# Modified to track paper source with each node
def baseline_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
  # The following does noun phrase extraction
  nlp = spacy.load("en_core_web_sm")
  doc = nlp(paper_text)
  noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks]

  # Reduce number of nodes, required for successful rendering
  noun_phrases = [np for np in noun_phrases if len(np.split()) > 1 and len(np) > 3]
  phrase_counts = Counter(noun_phrases)

  filtered_phrases = [phrase for phrase, _ in phrase_counts.most_common(node_limit)]
  nodes = list(set(filtered_phrases))
  # Return nodes with source information
  nodes_with_source = [(node, paper_source) for node in nodes]
  
  return nodes_with_source

#### summary nodes

In [113]:
# More summary focused method of creating nodes - modified to track paper source

def extract_sections(text):
    section_keywords = ['abstract', 'introduction', 'background', 'related work', 'method', 'methods',
                        'materials', 'experiments', 'results', 'discussion', 'conclusion', 'conclusions'] # won't put references, extra gibberish
    section_pattern = r'(?i)^(' + '|'.join(re.escape(k) for k in section_keywords) + r')$'

    lines = text.split('\n')
    sections = {}
    current_section = None
    for line in lines:
        stripped = line.strip().lower()
        if re.match(section_pattern, stripped):
            current_section = stripped
            print("Found section: " + current_section)
            sections[current_section] = []
        elif current_section:
            sections[current_section].append(stripped)

    # Join lines per section
    return {k: "\n".join(v).strip() for k, v in sections.items() if v}

def summarize_sections(sections, max_input_tokens=1500, max_output_tokens=300, summarizer=None):
    summarized = {}
    for name, content in sections.items():
        # Skip very short sections
        if len(content.strip().split()) < 10:
            continue

        # Truncate content to avoid token limit issues
        truncated = content.strip().replace("\n", " ")[:4000]

        try:
            summary = summarizer(
                truncated,
                max_length=max_output_tokens,
                min_length=100,
                do_sample=False
            )
            summarized[name] = summary[0]['summary_text']
        except Exception as e:
            print(f"Failed to summarize section '{name}': {e}")

    return summarized

def extract_phrases_from_summaries(summaries, phrases_per_section=20, kw_model=None):
    all_phrases = []
    for section, text in summaries.items():
        # Extract keywords/phrases using KeyBERT
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            use_maxsum=True,
            nr_candidates=20, # number of noun phrases returned for each summary
            top_n=phrases_per_section
        )
        section_phrases = [kw for kw, score in keywords]
        all_phrases.extend(section_phrases)

    return all_phrases

def get_top_nodes(phrases, limit=100):
    phrase_counts = Counter(phrases)
    return [phrase for phrase, _ in phrase_counts.most_common(limit)]

def summary_based_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
    summarizer = pipeline(
        "summarization",
        model="sshleifer/distilbart-cnn-12-6",  # You can change this to any supported summarization model
    )
    kw_model = KeyBERT(model='all-MiniLM-L6-v2')

    sections = extract_sections(paper_text)
    summaries = summarize_sections(sections, summarizer=summarizer)
    phrases = extract_phrases_from_summaries(summaries, phrases_per_section=20, kw_model=kw_model)

    if len(phrases) < node_limit:
        node_limit = len(phrases)

    nodes = get_top_nodes(phrases, node_limit)
    # Return nodes with source information
    nodes_with_source = [(node, paper_source) for node in nodes]
    return nodes_with_source

#### scientific_entity_create_nodes

In [114]:
# Improved scientific entity extraction with paper source tracking
def scientific_entity_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
    """
    Extract scientific entities using a transformer-based model and track their source
    
    Parameters:
    -----------
    paper_text : str
        The text content of the paper
    node_limit : int
        Maximum number of nodes to extract
    paper_source : str
        Source identifier (e.g., paper filename)
        
    Returns:
    --------
    list
        List of (node_text, paper_source) tuples
    """
    try:
        from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
        import torch
        
        print(f"Extracting scientific entities from paper: {paper_source}")
        
        # Use either SciBERT or BioBERT depending on the domain
        model_name = "allenai/scibert_scivocab_uncased"
        
        # Load model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(model_name)
        
        # Create NER pipeline
        ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
        
        # Process paper text in manageable chunks
        # SciBERT has a context window of 512 tokens, so we use slightly smaller chunks with overlap
        max_chunk_size = 450
        chunks = [paper_text[i:i+max_chunk_size] for i in range(0, len(paper_text), max_chunk_size-50)]
        
        all_entities = []
        for chunk in chunks[:50]:  # Limit chunks for practical purposes
            try:
                entities = ner_pipeline(chunk)
                all_entities.extend(entities)
            except Exception as e:
                print(f"Error processing chunk: {e}")
                continue
        
        # Extract entity texts and remove duplicates
        entity_texts = []
        for entity in all_entities:
            # Filter by confidence and length
            if entity['score'] > 0.7 and len(entity['word'].strip()) > 3:
                entity_texts.append(entity['word'].lower().strip())
        
        # Count entity occurrences and get the most frequent ones
        from collections import Counter
        entity_counter = Counter(entity_texts)
        top_entities = [entity for entity, _ in entity_counter.most_common(node_limit)]
        
        # Return nodes with source information
        nodes_with_source = [(entity, paper_source) for entity in top_entities]
        print(f"Extracted {len(nodes_with_source)} scientific entities")
        
        return nodes_with_source
        
    except ImportError as e:
        print(f"Error importing required packages: {e}")
        print("Falling back to baseline node creation method")
        return baseline_create_nodes(paper_text, node_limit, paper_source)

#### nltk_simple_nodes

In [115]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return ADJ
    elif treebank_tag.startswith('V'):
        return VERB
    elif treebank_tag.startswith('N'):
        return NOUN
    elif treebank_tag.startswith('R'):
        return ADV
    else:
        return NOUN

def nltk_simple_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
    # Tokenize and remove punctuation
    tokens = word_tokenize(paper_text.lower())
    tokens = [word for word in tokens if word.isalpha()]

    # Remove stop words
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # POS tagging for lemmatization
    tagger = PerceptronTagger()
    pos_tags = tagger.tag(filtered_tokens)

    # Lemmatize and stem tokens
    lemmatizer = WordNetLemmatizer()
    normalized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_tags]

    # Count frequency of words
    word_freq = Counter(normalized_tokens)

    # Extract most common keywords
    keywords = word_freq.most_common(node_limit)

    nodes_with_source = [(keyword, paper_source) for keyword, freq in keywords]

    # OPTIONAL LOGGING: keywords with their frequencies
    # for keyword, freq in keywords:
    #     print(f"{keyword}: {freq}")
    
    return nodes_with_source

#### gemini_create_nodes

In [116]:
def gemini_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
    """
    Extract high-level scientific concepts using the Gemini 2.5 Flash model (google-genai library, correct API).

    Parameters:
    -----------
    paper_text : str
        The text content of the paper.
    node_limit : int
        The maximum number of concepts to extract.
    paper_source : str
        The source identifier for the paper.

    Returns:
    --------
    list
        A list of (concept, paper_source) tuples.
    """
    if not gemini_api_key:
        print("Gemini API key is not configured. Skipping node creation.")
        return []

    try:
        model = "gemini-2.5-flash"
        prompt = f"""
        From the following research paper text, extract the top {node_limit} most important high-level scientific concepts, methods, and results.\nFocus on concepts that are central to the paper's contribution. Present them as a comma-separated list.\n\nPaper Text:\n\"\"\"\n{paper_text}\n\"\"\"
        """
        response = client.models.generate_content(
            model=model, contents=prompt
        )

        concepts = [concept.strip() for concept in response.text.split(',') if concept.strip()]
        nodes_with_source = [(concept, paper_source) for concept in concepts]
        return nodes_with_source
    except Exception as e:
        print(f"An error occurred with the Gemini API: {e}")
        return []

### create_nodes() -- unified function

In [117]:
NODE_CREATION_METHODS = {
    "baseline": baseline_create_nodes,
    "summary": summary_based_create_nodes,
    "scientific_entity": scientific_entity_create_nodes,
    "nltk_simple_nodes": nltk_simple_nodes,
    "gemini_create_nodes": gemini_create_nodes,
}

def create_nodes(
    paper_path,
    node_creation_method="summary",
    nodes_per_paper=20,
    **kwargs
):
    """
    Create nodes from a single paper.
    
    Parameters:
    -----------
    paper_path : str
        Path to the PDF or text file of the paper
    node_creation_method : str
        The node creation method to use (must be in NODE_CREATION_METHODS)
    nodes_per_paper : int
        Number of nodes to extract
    kwargs : dict
        Additional keyword arguments for the node creation function
    
    Returns:
    --------
    nodes_with_source : list
        List of (concept, paper_source) tuples
    """
    # Load paper text (assume load_paper_text_from_file is defined)
    paper_text = load_paper_text_from_file(paper_path)
    
    # Get the node creation function
    func = NODE_CREATION_METHODS.get(node_creation_method)
    if func is None:
        raise ValueError(f"Unknown node creation method: {node_creation_method}")
    
    # Call the node creation function
    nodes_with_source = func(paper_text, nodes_per_paper, paper_path, **kwargs)
    
    # Print results
    print(f"\nExtracted {len(nodes_with_source)} nodes using '{node_creation_method}' from '{paper_path}':")
    for i, (concept, source) in enumerate(nodes_with_source, 1):
        print(f"{i:2d}. {concept}")
    return nodes_with_source

### TEST: Node Creation Methods

In [118]:
# paper_paths = [
#     "pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf",
#     # "pdfs/biology/Transfer learning improves performance in volumetric.pdf",
#     # "pdfs/emotions/Can MOOC Instructor Be Portrayed by Semantic Features.pdf",
#     # "pdfs/emotions/Learners Performance in a MOOC on Programming.pdf",
# ]

# # Example usage
# for paper_path in paper_paths:
#     print(f"\nTesting node creation on paper: {paper_path}")
#     nodes = create_nodes(
#         paper_path,
#         node_creation_method="gemini_create_nodes",
#         nodes_per_paper=50
#     )

#### summary nodes test: different models

##### default model

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.

Extracted 40 nodes using 'summary' from 'pdfs/biology/Transfer learning improves performance in volumetric.pdf':
 1. microscopy
 2. electron microscopy
 3. volumetric electron microscopy
 4. transfer learning
 5. liver dataset
 6. volumetric electron
 7. rat liver dataset
 8. microscopy vem
 9. electron microscopy vem
10. liver dataset mouseliver
11. cells structures image
12. identification labeling organelles
13. tissues organelle
14. nanoscale resolution dimensional
15. dimensional 3d imaging
16. imaging biological samples
17. labeling organelles
18. mammalian tissues organelle
19. microscopy vem enables
20. labeling organelles cells
21. 3d imaging biological
22. liver dataset imaged
23. array tomography
24. cellular structures 3d
25. images cellular
26. structures 3d tissue
27. array tomography 22
28. resolution images cellular
29. nanoscale resolution
30. 3d tissue
31. generates nanoscale resolution
32. vem generates nanoscale
33. 3d tissue scale
34. nanoscale resolution images
35. images cellular structures
36. microscopy vem generates
37. scenarios transfer learning
38. training data scarce
39. pretraining task mouse
40. 10 simulate training
Nodes extracted: 40

##### philschmid/bart-large-cnn-samsum

Device set to use cpu
Found section: abstract
Found section: introduction
Found section: results
Found section: discussion
Found section: method
Found section: experiments
Found section: methods
Your max_length is set to 300, but your input_length is only 266. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=133)

Extracted 40 nodes using 'summary' from 'pdfs/biology/Transfer learning improves performance in volumetric.pdf':
 1. transfer learning
 2. microscopy
 3. electron microscopy
 4. volumetric electron microscopy
 5. organelle
 6. volumetric electron
 7. microscopy vem
 8. electron microscopy vem
 9. liver dataset mouseliver
10. organelles
11. training transfer learning
12. deep learning
13. automated using deep
14. cells structures image
15. tissues organelle
16. microscopy vem enables
17. 3d imaging biological
18. mammalian tissues organelle
19. deep learning segmentation
20. labeling organelles
21. manual labeling organelles
22. labeling organelles cells
23. bladder mouse cortex
24. train 3d net
25. resolution images cellular
26. 3d resu
27. images cellular
28. manually transfer learning
29. microscopy vem generates
30. 3d resu net
31. images cellular structures
32. cellular structure label
33. 3d tissue
34. structures 3d tissue
35. 3d tissue scale
36. cellular structures 3d
37. weights using images
38. mitochondria segmentation
39. data pretrain
40. pretraining task mouse
Nodes extracted: 40

### gemini-2.5-flash

Extracted 52 nodes using 'gemini_create_nodes' from 'pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf':
 1. Automated cell structure extraction
 2. 3D electron microscopy (EM)
 3. Deep learning-based segmentation
 4. Focused ion beam scanning electron microscope (FIB-SEM)
 5. U-Net (neural network architecture)
 6. Segment Anything Model (SAM)
 7. 3D watershed algorithm
 8. Sequential cross-sectional images (SBF data)
 9. Cyanidioschyzon merolae (model organism)
10. Intracellular organelle segmentation
11. Manual data labeling (expert annotation)
12. Image preprocessing techniques
13. Noise reduction (e.g.
14. median
15. bilateral filters)
16. Image aspect ratio correction
17. Image normalization
18. Cross-validation (model training and evaluation)
19. Image data augmentation (multi-directional views)
20. Scanning 3D U-Net architecture
21. Skip connections (in U-Net)
22. Attention gate mechanism
23. ReLU activation function
24. Batch normalization
25. TverskyLoss function
26. AdamW optimizer
27. Intersection over Union (IoU) (evaluation metric)
28. High background segmentation accuracy
29. High plastid segmentation accuracy
30. Moderate mitochondrion segmentation accuracy
31. Moderate nucleus segmentation accuracy
32. Low peroxisome segmentation accuracy
33. Cell division stage dependency on accuracy
34. Multi-directional inference (for accuracy improvement)
35. Voxel-level probability determination
36. Full automation of 3D cell model creation
37. Extraction of individual cells from multi-cell images
38. Zero-shot capabilities (of SAM)
39. Bottleneck of extensive labeled data
40. Applicability to diverse 3D imaging devices
41. Objective and quantitative biological image analysis
42. Morphological analysis of cellular structures
43. Dynamic analysis of cellular structures
44. Single-channel grayscale image processing (EM images)
45. Complex cell structure boundaries (challenge)
46. Low contrast images (challenge)
47. High-pressure freezing (sample preparation)
48. Cell cycle synchronization
49. Cell division mechanisms investigation
50. Automated 3D reconstruction pipeline
51. Addressing segmentation subjectivity
52. Automated identification of specific cells in mixed environments

## Step 5: Creation of Edges

### Edge Creation Methods

#### create_edges_by_sentence_coocurrence

In [119]:
# Modified to work with nodes that include source information
def create_edges_by_sentence_coocurrence(nodes_with_source, papers_dict):
  nlp = spacy.load("en_core_web_sm")
  edge_weights = defaultdict(int)

  # Extract just the node texts for matching
  node_texts = [node[0] for node in nodes_with_source]
  
  # Process each paper
  for paper_path, paper_text in papers_dict.items():
    doc = nlp(paper_text)
    sentences = [sent.text.lower() for sent in doc.sents]
    
    for sentence in sentences:
      # Find which nodes are present in this sentence
      present_nodes = [node_idx for node_idx, (node, _) in enumerate(nodes_with_source) if node in sentence]
      for idx1, idx2 in itertools.combinations(set(present_nodes), 2):
        # Use indices to get the original nodes with source
        node1, node2 = nodes_with_source[idx1], nodes_with_source[idx2]
        # Sort by node text to ensure consistent edge representation
        if node1[0] > node2[0]:
            node1, node2 = node2, node1
        edge = (node1, node2)
        edge_weights[edge] += 1

  edges = [(a, b, {'weight': w}) for (a, b), w in edge_weights.items()]
  return edges

#### create_edges_by_summary_cooccurrence

In [120]:
def create_edges_by_summary_cooccurrence(nodes_with_source, papers_dict):
    edges = defaultdict(int)
    
    # For each paper, extract sections and summaries
    for paper_path, paper_text in papers_dict.items():
        sections = extract_sections(paper_text)
        summaries = summarize_sections(sections)
        
        for summary in summaries.values():
            summary_lower = summary.lower()
            # Find which nodes are present in this summary
            present_nodes = [node_with_source for node_with_source in nodes_with_source 
                            if node_with_source[0] in summary_lower]
            
            for node1, node2 in combinations(set(present_nodes), 2):
                # Sort by node text to ensure consistent edge representation
                if node1[0] > node2[0]:
                    node1, node2 = node2, node1
                edges[(node1, node2)] += 1  # Increment weight if they co-occur

    return [(a, b, {'weight': w}) for (a, b), w in edges.items()]

#### create_edges_by_embedding_similarity

In [121]:
def create_edges_by_embedding_similarity(nodes_with_source, threshold=0.5):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # Extract just the node texts for embedding
    node_texts = [node[0] for node in nodes_with_source]
    embeddings = model.encode(node_texts, convert_to_tensor=True)

    edges = []
    for i in range(len(nodes_with_source)):
        for j in range(i + 1, len(nodes_with_source)):
            sim = util.cos_sim(embeddings[i], embeddings[j]).item()
            if sim >= threshold:
                edges.append((nodes_with_source[i], nodes_with_source[j], {'weight': sim}))
    return edges

#### create_edges_by_relation_extraction

In [122]:
def create_edges_by_relation_extraction(nodes_with_source, papers_dict):
    """
    Extract edges between nodes based on subject-verb-object relationships
    
    Parameters:
    -----------
    nodes_with_source : list
        List of (node_text, source) tuples
    papers_dict : dict
        Dictionary mapping paper paths to paper text content
        
    Returns:
    --------
    list
        List of edges with relation information
    """
    try:
        import spacy
        from spacy.matcher import DependencyMatcher
        print("Creating edges using relation extraction...")
        
        # Load spaCy model
        nlp = spacy.load("en_core_web_sm")
        
        # Create a dependency matcher for subject-verb-object patterns
        matcher = DependencyMatcher(nlp.vocab)
        
        # Define SVO pattern
        pattern = [
            # Subject
            {
                "RIGHT_ID": "subject",
                "RIGHT_ATTRS": {"DEP": {"IN": ["nsubj", "nsubjpass"]}}
            },
            # Verb
            {
                "LEFT_ID": "subject",
                "REL_OP": ">",
                "RIGHT_ID": "verb",
                "RIGHT_ATTRS": {"POS": "VERB"}
            },
            # Object
            {
                "LEFT_ID": "verb",
                "REL_OP": ">",
                "RIGHT_ID": "object",
                "RIGHT_ATTRS": {"DEP": {"IN": ["dobj", "pobj"]}}
            }
        ]
        
        matcher.add("SVO", [pattern])
        
        # Extract just the node texts for matching
        node_texts = [node[0] for node in nodes_with_source]
        node_dict = {node[0]: i for i, node in enumerate(nodes_with_source)}
        
        edges = defaultdict(list)
        
        # Process each paper
        for paper_path, paper_text in papers_dict.items():
            print(f"Extracting relations from: {paper_path}")
            
            # Process the text in chunks 
            chunk_size = 5000  # Reduce memory usage
            for i in range(0, min(len(paper_text), 100000), chunk_size):  # Limit for practical demo
                chunk = paper_text[i:i+chunk_size]
                try:
                    doc = nlp(chunk)
                    
                    # Find SVO triples
                    matches = matcher(doc)
                    
                    for match_id, token_ids in matches:
                        subject = doc[token_ids[0]].text.lower()
                        verb = doc[token_ids[1]].text.lower()
                        obj = doc[token_ids[2]].text.lower()
                        
                        # Find nodes that contain our subject and object
                        subject_nodes = [n for n in node_texts if subject in n]
                        object_nodes = [n for n in node_texts if obj in n]
                        
                        # Create edges between matching subject and object nodes
                        for subj_node in subject_nodes:
                            for obj_node in object_nodes:
                                if subj_node != obj_node:
                                    subj_idx = node_dict[subj_node]
                                    obj_idx = node_dict[obj_node]
                                    
                                    # Create node pair key for the edge
                                    node1 = nodes_with_source[subj_idx]
                                    node2 = nodes_with_source[obj_idx]
                                    
                                    if node1[0] > node2[0]:  # Ensure consistent ordering
                                        node1, node2 = node2, node1
                                        
                                    edge_key = (node1, node2)
                                    edges[edge_key].append(verb)
                                    
                except Exception as e:
                    print(f"Error processing chunk: {e}")
                    continue
        
        # Convert edges dict to list format with relations
        edge_list = []
        for (node1, node2), relations in edges.items():
            # Count relation occurrences
            relation_counter = Counter(relations)
            most_common_relation = relation_counter.most_common(1)[0][0]
            weight = len(relations)  # Use count of relations as weight
            
            edge_list.append((
                node1, 
                node2, 
                {'weight': weight, 'relation': most_common_relation}
            ))
            
        print(f"Created {len(edge_list)} edges with relation information")
        return edge_list
        
    except Exception as e:
        print(f"Error in relation extraction: {e}")
        print("Falling back to co-occurrence based edge creation")
        return create_edges_by_sentence_coocurrence(nodes_with_source, papers_dict)

#### create_edges_with_gemini

In [123]:
def create_edges_by_gemini(nodes_with_source, papers_dict, model_name="models/gemini-2.5-flash"):
    """
    Use Gemini to suggest edges (relationships) between nodes based on their meaning and context.
    Each edge will be a tuple: (node1, node2, {"weight": 1.0, "relation": relation_string})
    """
    if not gemini_api_key:
        print("Gemini API key is not configured. Skipping Gemini edge creation.")
        return []
    try:
        model = client.get_model(name=model_name)
        # Prepare a prompt with all node names and optionally some context
        node_texts = [node[0] for node in nodes_with_source]
        node_list_str = "\n".join(f"- {n}" for n in node_texts)
        prompt = f"""
        Given the following list of scientific concepts/nodes from a research paper knowledge graph, suggest the most meaningful relationships (edges) between them. For each edge, provide the two node names and a short relation label (e.g., 'is_part_of', 'enables', 'related_to', etc). Only include edges that are meaningful and non-trivial.
        
        Nodes:\n{node_list_str}
        """
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": List[GeminiEdge],
            },
        )
        # Use instantiated objects (parsed)
        gemini_edges: List[GeminiEdge] = response.parsed
        edges = []
        for edge in gemini_edges:
            node1_full = next((n for n in nodes_with_source if n[0] == edge.node1), None)
            node2_full = next((n for n in nodes_with_source if n[0] == edge.node2), None)
            if node1_full and node2_full and node1_full != node2_full:
                edges.append((node1_full, node2_full, {"weight": 1.0, "relation": edge.relation}))
        print(f"Gemini (structured) suggested {len(edges)} edges.")
        return edges
    except Exception as e:
        print(f"An error occurred with Gemini edge creation (structured): {e}")
        return []

### create_edges() -- unified function

In [124]:
# Unified Edge Creation Methods
EDGE_CREATION_METHODS = {
    "create_edges_by_sentence_coocurrence": create_edges_by_sentence_coocurrence,
    "create_edges_by_summary_cooccurrence": create_edges_by_summary_cooccurrence,
    "create_edges_by_embedding_similarity": create_edges_by_embedding_similarity,
    "create_edges_by_relation_extraction": create_edges_by_relation_extraction,
    "create_edges_by_gemini": create_edges_by_gemini,
}

def create_edges(
    nodes_with_source,
    papers_dict=None,
    edge_creation_method="create_edges_by_embedding_similarity",
    threshold=0.5,
    **kwargs
):
    """
    Create edges between nodes using the specified method.
    
    Parameters:
    -----------
    nodes_with_source : list
        List of (node_text, paper_source) tuples
    papers_dict : dict or None
        Dictionary mapping paper paths to text (required for some methods)
    edge_creation_method : str
        The edge creation method to use (must be in EDGE_CREATION_METHODS)
    threshold : float
        Similarity threshold for embedding-based methods
    kwargs : dict
        Additional keyword arguments for the edge creation function
    
    Returns:
    --------
    edges : list
        List of edges (node1, node2, attributes)
    """
    func = EDGE_CREATION_METHODS.get(edge_creation_method)
    if func is None:
        raise ValueError(f"Unknown edge creation method: {edge_creation_method}")
    
    # Some methods require papers_dict, others do not
    if edge_creation_method == "create_edges_by_embedding_similarity":
        edges = func(nodes_with_source, threshold=threshold, **kwargs)
    else:
        edges = func(nodes_with_source, papers_dict, **kwargs)
    
    print(f"\nExtracted {len(edges)} edges using '{edge_creation_method}' method.")
    return edges

### TEST: Edge Creation Methods

In [ ]:
# paper_paths = [
#     "pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf",
#     # "pdfs/biology/Transfer learning improves performance in volumetric.pdf",
#     # "pdfs/emotions/Can MOOC Instructor Be Portrayed by Semantic Features.pdf",
#     # "pdfs/emotions/Learners Performance in a MOOC on Programming.pdf",
# ]

# nodes = []
# for paper_path in paper_paths:
#     print(f"\nTesting node creation on paper: {paper_path}")
#     nodes = create_nodes(
#         paper_path,
#         node_creation_method="gemini_create_nodes",
#         nodes_per_paper=50
#     )


Testing node creation on paper: pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf

Extracted 49 nodes using 'gemini_create_nodes' from 'pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf':
 1. Automated Cell Structure Extraction
 2. 3D Electron Microscopy (EM)
 3. Deep Learning (DL)
 4. 3D Cell Structure Modeling
 5. Sequential Cross-Sectional Images
 6. Image Segmentation
 7. U-Net Neural Network
 8. Segment Anything Model (SAM)
 9. 3D Watershed Algorithm
10. Focused Ion Beam Scanning Electron Microscope (FIB-SEM)
11. Cyanidioschyzon merolae (Model Organism)
12. Cell Organelle Segmentation
13. Intracellular Structure Information
14. Laborious Manual Segmentation
15. Bottleneck in Biological Image Analysis
16. Serial Block-Face (SBF) Image Data
17. Image Preprocessing
18. Noise Removal (Image Processing)
19. Data Normalization (Image Processing)
20. Data Augmentation (Multi-Directional Views)
21. Convolutional Layers
22. Deconvolutional Layers
23. 

In [ ]:
# papers_dict = {paper_path: load_paper_text_from_file(paper_path) for paper_path in paper_paths}
# # print(papers_dict)
# # create edges
# edges = create_edges(
#     nodes_with_source=nodes,
#     papers_dict=papers_dict,
#     edge_creation_method="create_edges_by_embedding_similarity",
# )
# # Print edges
# print("\nExtracted Edges:")
# for i, (node1, node2, attrs) in enumerate(edges, 1):
#     print(f"{i:2d}. {node1[0]} -- {node2[0]} (weight: {attrs['weight']}, relation: {attrs.get('relation', 'N/A')})")



Extracted 74 edges using 'create_edges_by_embedding_similarity' method.

Extracted Edges:
 1. Automated Cell Structure Extraction -- 3D Cell Structure Modeling (weight: 0.6560109853744507, relation: N/A)
 2. Automated Cell Structure Extraction -- Cell Organelle Segmentation (weight: 0.658442497253418, relation: N/A)
 3. Automated Cell Structure Extraction -- Intracellular Structure Information (weight: 0.5955420732498169, relation: N/A)
 4. Automated Cell Structure Extraction -- Mitochondrion Segmentation (weight: 0.5040507912635803, relation: N/A)
 5. Automated Cell Structure Extraction -- Nuclei Segmentation (weight: 0.5000455975532532, relation: N/A)
 6. Automated Cell Structure Extraction -- Cytoplasm Segmentation (weight: 0.543447732925415, relation: N/A)
 7. Automated Cell Structure Extraction -- Cell Division Stages (weight: 0.5144004225730896, relation: N/A)
 8. Automated Cell Structure Extraction -- Automated Individual Cell Extraction (weight: 0.8533382415771484, relation: N

## Step 6: Populate Graph

In [67]:
# Step 6: Populate graph - modified to work with nodes that include source information

def create_graph(nodes_with_source, edges, paper_colors=None):
  graph = nx.Graph()
  
  # If no colors provided, generate random distinct colors
  if paper_colors is None:
    import random
    all_sources = set(source for _, source in nodes_with_source)
    paper_colors = {source: f"#{random.randint(0, 0xFFFFFF):06x}" for source in all_sources}

  # Add nodes with source information as attributes
  for node_text, source in nodes_with_source:
    graph.add_node(node_text, source=source, color=paper_colors.get(source, "#808080"), 
                   title=f"Source: {source}")

  # Add edges
  for (node1_with_source, node2_with_source, weight) in edges:
    node1_text, _ = node1_with_source
    node2_text, _ = node2_with_source
    graph.add_edge(node1_text, node2_text, weight=weight['weight'])

  print(f"Number of nodes: {len(graph.nodes)}")
  print(f"Number of edges: {len(graph.edges)}\n")
  print(f"Sources: {set(nx.get_node_attributes(graph, 'source').values())}")

  return graph

## Steps 7 & 8: Write Out And Display Graph

In [68]:
def graph_to_html(graph, path: str, display: False):
    net = Network(height="750px", width="100%", notebook=True, cdn_resources="in_line")
    
    # Configure network to show node information on hover
    net.set_options("""
    {
      "nodes": {
        "font": {
          "size": 15,
          "face": "Tahoma"
        }
      },
      "edges": {
        "color": {
          "inherit": true
        },
        "smooth": false
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -80000,
          "springLength": 250,
          "springConstant": 0.001
        },
        "minVelocity": 0.75
      }
    }
    """)
    
    # Add nodes with source information
    for node, data in graph.nodes(data=True):
        title = f"Source: {data.get('source', 'Unknown')}"
        net.add_node(node, color=data.get('color', '#808080'), title=title, label=node)
    
    # Add edges
    for source, target, data in graph.edges(data=True):
        net.add_edge(source, target, value=data.get('weight', 1))
    
    # Write HTML with UTF-8 encoding to avoid UnicodeEncodeError on Windows
    html_str = net.generate_html()
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_str)

    if display:
        from IPython.display import HTML
        with open(path, "r", encoding="utf-8") as f:
            html_content = f.read()
        display(HTML(html_content))

In [69]:
# Enhanced graph visualization with relationship labels and interactivity
def advanced_graph_to_html(graph, path: str, display: False):
    """
    Create an enhanced interactive HTML visualization of the knowledge graph
    
    Parameters:
    -----------
    graph : networkx.Graph
        The knowledge graph
    path : str
        Path to save the HTML file
    display : bool
        Whether to display the graph in the notebook
    """
    net = Network(height="750px", width="100%", notebook=True, cdn_resources="in_line")
    
    # Configure advanced network options for better visualization
    net.set_options("""
    {
      "nodes": {
        "font": {"size": 14, "face": "Tahoma"},
        "scaling": {
          "min": 10,
          "max": 30,
          "label": {
            "enabled": true,
            "min": 14,
            "max": 30
          }
        },
        "shape": "dot"
      },
      "edges": {
        "font": {"size": 12, "align": "middle"},
        "color": {"inherit": "both"},
        "smooth": {"type": "continuous", "roundness": 0.5},
        "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}},
        "scaling": {"min": 1, "max": 10}
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 200,
        "hideEdgesOnDrag": false,
        "navigationButtons": true
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -80000,
          "springLength": 250,
          "springConstant": 0.001,
          "damping": 0.09
        },
        "minVelocity": 0.75
      }
    }
    """)
    
    # Add nodes with source information and size based on degree
    degrees = dict(graph.degree())
    max_degree = max(degrees.values()) if degrees else 1
    
    for node, data in graph.nodes(data=True):
        # Scale node size based on its degree centrality
        size = 10 + (degrees.get(node, 1) / max_degree) * 20
        
        # Create informative tooltip
        source = data.get('source', 'Unknown')
        paper_name = source.split('/')[-1]
        title = f"Concept: {node}<br>Source: {paper_name}<br>Connections: {degrees.get(node, 0)}"
        
        net.add_node(
            node, 
            color=data.get('color', '#808080'), 
            title=title, 
            label=node,
            size=size
        )
    
    # Add edges with relationship information when available
    for source, target, data in graph.edges(data=True):
        if 'relation' in data:
            # For edges with extracted relations
            weight = data.get('weight', 1)
            relation = data.get('relation', '')
            title = f"{relation} (weight: {weight})"
            
            net.add_edge(
                source, target, 
                value=min(weight, 10),  # Cap weight for visualization
                title=title,
                label=relation
            )
        else:
            # For edges without relation information
            weight = data.get('weight', 1)
            net.add_edge(source, target, value=min(weight, 10), title=f"weight: {weight}")
    
    # Add legend for paper sources
    paper_sources = set(nx.get_node_attributes(graph, 'source').values())
    legend_html = "<div style='position:absolute; top: 10px; left: 10px; background-color: rgba(255,255,255,0.8); padding: 10px; border-radius: 5px;'>"
    legend_html += "<h3>Paper Sources</h3>"
    
    for source in paper_sources:
        paper_name = source.split('/')[-1]
        color = next((data.get('color') for _, data in graph.nodes(data=True) if data.get('source') == source), '#808080')
        legend_html += f"<div><span style='background-color:{color}; width:15px; height:15px; display:inline-block; margin-right:5px;'></span>{paper_name}</div>"
    
    legend_html += "</div>"
    
    # Add network statistics
    stats_html = "<div style='position:absolute; bottom: 10px; left: 10px; background-color: rgba(255,255,255,0.8); padding: 10px; border-radius: 5px;'>"
    stats_html += f"<div><b>Total nodes:</b> {len(graph.nodes)}</div>"
    stats_html += f"<div><b>Total edges:</b> {len(graph.edges)}</div>"
    stats_html += "</div>"
    
    # Write HTML with UTF-8 encoding
    html_str = net.generate_html()
    # Insert legend and stats before the closing body tag
    html_str = html_str.replace("</body>", f"{legend_html}{stats_html}</body>")
    
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_str)

    if display:
        from IPython.display import HTML
        display(HTML(html_str))
        
    print(f"Enhanced visualization saved to: {path}")

# Complete Multi-Paper Knowledge Graph Generator

In [70]:
# # Define paper paths
# paper_paths = [
#     "pdfs/Transfer learning improves performance in volumetric.pdf",
#     "pdfs/Automated Cell Structure Extraction for 3D Electron.pdf"
# ]

# # Define colors for each paper
# paper_colors = {
#     paper_paths[0]: "#3366CC",  # Blue for first paper
#     paper_paths[1]: "#DC3912"   # Red for second paper
# }

# # Load papers
# papers_dict = load_multiple_papers(paper_paths)

# # Create nodes for each paper (limit nodes per paper to avoid overcrowding)
# nodes_per_paper = 25
# combined_nodes = []

# # Option 1: Baseline node creation
# # for paper_path, paper_text in papers_dict.items():
# #     paper_nodes = baseline_create_nodes(paper_text, nodes_per_paper, paper_path)
# #     combined_nodes.extend(paper_nodes)
# #     print(f"Created {len(paper_nodes)} nodes from {paper_path}")

# # Option 2: Summary-based node creation (comment out Option 1 if using this)
# summarizer = pipeline("summarization")
# kw_model = KeyBERT()
# for paper_path, paper_text in papers_dict.items():
#     sections = extract_sections(paper_text)
#     summaries = summarize_sections(sections)
#     paper_nodes = summary_based_create_nodes(paper_text, nodes_per_paper, paper_path)
#     combined_nodes.extend(paper_nodes)
#     print(f"Created {len(paper_nodes)} nodes from {paper_path}")

# # Create edges
# # You can choose one of these edge creation methods
# # edges = create_edges_by_sentence_coocurrence(combined_nodes, papers_dict)
# # edges = create_edges_by_summary_cooccurrence(combined_nodes, papers_dict)
# edges = create_edges_by_embedding_similarity(combined_nodes, threshold=0.6)

# # Create graph
# multi_paper_graph = create_graph(combined_nodes, edges, paper_colors)

# # Export graph to HTML
# graph_to_html(multi_paper_graph, "multi_paper_knowledge_graph_25_nodes.html", False)

# print(f"Total nodes: {len(combined_nodes)}")
# print(f"Total edges: {len(edges)}")

## Analysis Functions for Multi-Paper Knowledge Graphs

In [71]:
def analyze_multi_paper_graph(graph):
    """Analyze the multi-paper knowledge graph to identify connections between papers"""
    # Get node sources
    node_sources = nx.get_node_attributes(graph, 'source')
    
    # Count nodes by source
    source_counts = Counter(node_sources.values())
    print("Nodes per paper:")
    for source, count in source_counts.items():
        print(f"  {source}: {count} nodes")
    
    # Find cross-paper connections
    cross_paper_edges = []
    for u, v in graph.edges():
        if node_sources[u] != node_sources[v]:
            cross_paper_edges.append((u, v))
    
    print(f"\nCross-paper connections: {len(cross_paper_edges)} edges")
    if cross_paper_edges:
        print("Sample cross-paper connections:")
        for i, (u, v) in enumerate(cross_paper_edges[:5]):  # Show up to 5 examples
            print(f"  {u} ({node_sources[u]}) <--> {v} ({node_sources[v]})")
    
    # Find highest-degree nodes
    degrees = dict(graph.degree())
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop connected concepts:")
    for node, degree in top_nodes:
        print(f"  {node} ({node_sources[node]}): {degree} connections")
            
    return {
        'source_counts': source_counts,
        'cross_paper_edges': cross_paper_edges,
        'top_nodes': top_nodes
    }

# Example usage
# analysis = analyze_multi_paper_graph(multi_paper_graph)

# Knowledge Graph Generator - Unified API

In [ ]:
def generate_knowledge_graph(
    paper_paths,
    nodes_per_paper=25,
    node_creation_method='baseline',
    edge_creation_method='embedding_similarity',
    threshold=0.6,
    output_path=None,
    display=False,
    custom_colors=None,
    advanced_visualization=False,
):
    """
    Generate a knowledge graph from multiple research papers
    
    Parameters:
    -----------
    paper_paths : list
        List of paths to PDF files
    nodes_per_paper : int
        Number of nodes to extract per paper
    node_creation_method : str
        Method to use for node creation ('baseline', 'summary', 'scientific_entity', or 'high_level_concepts')
    edge_creation_method : str
        Method to use for edge creation ('sentence_coocurrence', 'summary_cooccurrence', 
                                        'embedding_similarity', 'relation_extraction', or 'scientific_connections')
    threshold : float
        Similarity threshold for embedding-based edge creation
    output_path : str
        Path to save the HTML file. If None, a descriptive name will be generated
    display : bool
        Whether to display the graph in the notebook
    custom_colors : dict
        Custom colors for each paper (paper_path -> color)
    advanced_visualization : bool
        Whether to use the advanced visualization with relationship labels
    
    Returns:
    --------
    graph : networkx.Graph
        The generated knowledge graph
    """
    # 1. Load papers
    print(f"Loading {len(paper_paths)} papers...")
    papers_dict = load_multiple_papers(paper_paths)
    if not papers_dict:
        print("No papers were successfully loaded. Exiting.")
        return None
    
    # Define colors for each paper if not provided
    if custom_colors is None:
        import random
        # Use distinctive colors rather than random ones
        distinctive_colors = [
            "#4285F4",  # Google Blue
            "#EA4335",  # Google Red
            "#FBBC05",  # Google Yellow
            "#34A853",  # Google Green
            "#FF9900",  # Amazon Orange
            "#146EB4",  # Walmart Blue
            "#00A1E0",  # Salesforce Blue
            "#0F9D58",  # Android Green
            "#AB47BC",  # Purple
            "#00BCD4",  # Cyan
            "#FF5722",  # Deep Orange
            "#795548"   # Brown
        ]
        paper_colors = {}
        for i, path in enumerate(paper_paths):
            paper_colors[path] = distinctive_colors[i % len(distinctive_colors)]
    else:
        paper_colors = custom_colors
    
    # 2. Create nodes for each paper
    print(f"Creating nodes using {node_creation_method} method ({nodes_per_paper} nodes per paper)...")
    combined_nodes = []
    
    # Initialize models if needed
    if node_creation_method == 'summary':
        summarizer = pipeline("summarization")
        kw_model = KeyBERT()
    
    # Create nodes for each paper
    for paper_path, paper_text in papers_dict.items():
        paper_nodes = create_nodes(paper_path, node_creation_method, nodes_per_paper)
        combined_nodes.extend(paper_nodes)
        print(f"  Created {len(paper_nodes)} nodes from {paper_path}")
    
    # 3. Create edges
    print(f"Creating edges using {edge_creation_method} method...")
    if edge_creation_method == 'sentence_coocurrence':
        edges = create_edges_by_sentence_coocurrence(combined_nodes, papers_dict)
    elif edge_creation_method == 'summary_cooccurrence':
        edges = create_edges_by_summary_cooccurrence(combined_nodes, papers_dict)
    elif edge_creation_method == 'embedding_similarity':
        edges = create_edges_by_embedding_similarity(combined_nodes, threshold)
    elif edge_creation_method == 'relation_extraction':
        edges = create_edges_by_relation_extraction(combined_nodes, papers_dict)
    else:
        raise ValueError(f"Unknown edge creation method: {edge_creation_method}")
    
    print(f"  Created {len(edges)} edges")
    
    # 4. Create graph
    print("Building and visualizing the knowledge graph...")
    knowledge_graph = create_graph(combined_nodes, edges, paper_colors)
    
    # 5. Generate descriptive output path if not provided
    total_nodes = len(knowledge_graph.nodes)
    if output_path is None:
        output_path = f"kg_{node_creation_method}_{edge_creation_method}_{total_nodes}_nodes.html"
    
    # 6. Export graph to HTML - choose between standard and advanced visualization
    if advanced_visualization:
        advanced_graph_to_html(knowledge_graph, output_path, display)
    else:
        graph_to_html(knowledge_graph, output_path, display)
    
    print(f"Knowledge graph generated and saved to {output_path}")
    print(f"Total nodes: {total_nodes}")
    print(f"Total edges: {len(edges)}")
    
    return knowledge_graph

### compare_knowledge_graph_methods()

In [73]:
def compare_knowledge_graph_methods(paper_paths, nodes_per_paper=30):
    """
    Generate and compare knowledge graphs using different methods
    
    Parameters:
    -----------
    paper_paths : list
        List of paths to PDF files
    nodes_per_paper : int
        Number of nodes to extract per paper
    """
    import time
    import pandas as pd
    
    methods = [
        {
            "name": "Baseline + Co-occurrence",
            "node_method": "baseline",
            "edge_method": "sentence_coocurrence"
        },
        {
            "name": "Summary + Embedding",
            "node_method": "summary",
            "edge_method": "embedding_similarity"
        },
        {
            "name": "Scientific Entity + Relation",
            "node_method": "scientific_entity",
            "edge_method": "relation_extraction"
        }
    ]
    
    results = []
    
    for method in methods:
        print(f"\n\n{'='*50}")
        print(f"Testing: {method['name']}")
        print(f"{'='*50}")
        
        start_time = time.time()
        
        # Generate graph
        kg = generate_knowledge_graph(
            paper_paths=paper_paths,
            nodes_per_paper=nodes_per_paper,
            node_creation_method=method["node_method"],
            edge_creation_method=method["edge_method"],
            advanced_visualization=(method["edge_method"] == "relation_extraction")
        )
        
        # Collect metrics
        if kg:
            end_time = time.time()
            execution_time = end_time - start_time
            
            # Get cross-paper connections
            analysis = analyze_multi_paper_graph(kg)
            
            results.append({
                "Method": method["name"],
                "Nodes": len(kg.nodes),
                "Edges": len(kg.edges),
                "Density": nx.density(kg),
                "Cross-paper Connections": len(analysis["cross_paper_edges"]),
                "Execution Time (s)": round(execution_time, 2)
            })
    
    # Create comparison table
    if results:
        results_df = pd.DataFrame(results)
        display(results_df)
        
        # Save comparison to CSV
        results_df.to_csv("knowledge_graph_method_comparison.csv", index=False)
        print("Comparison results saved to knowledge_graph_method_comparison.csv")
    else:
        print("No results generated for comparison")

# Uncomment to run comparison
# compare_knowledge_graph_methods(paper_paths)

## Example Using the New High-Level Concept Extraction

In [74]:
paper_paths = [
    "pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf",
    "pdfs/biology/Transfer learning improves performance in volumetric.pdf",
    "pdfs/emotions/Can MOOC Instructor Be Portrayed by Semantic Features.pdf",
    "pdfs/emotions/Learners Performance in a MOOC on Programming.pdf"
]

# Custom colors for visual distinction
custom_colors = {
    paper_paths[0]: "#4285F4",  # Google Blue
    paper_paths[1]: "#EA4335"   # Google Red
}

In [ ]:
paper_paths = [
    "pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf"
]

# Generate the knowledge graph with high-level scientific concepts
scientific_graph = generate_knowledge_graph(
    paper_paths=paper_paths,
    # nodes_per_paper=15,  # Fewer, higher-quality concepts
    node_creation_method='gemini_create_nodes',
    edge_creation_method='embedding_similarity',
    # threshold=0.65,
    custom_colors=custom_colors,
    advanced_visualization=True
)

# Analyze the results
if scientific_graph:
    analysis = analyze_multi_paper_graph(scientific_graph)
    
    # Get details about the relation types
    relation_types = set()
    for _, _, data in scientific_graph.edges(data=True):
        if 'relation' in data:
            relation_types.add(data['relation'])
    
    print("\nDetected relation types:")
    print(", ".join(sorted(relation_types)))
    
    # Count connections by type
    relation_counts = Counter()
    for _, _, data in scientific_graph.edges(data=True):
        relation = data.get('relation', 'unknown')
        relation_counts[relation] += 1
    
    print("\nConnection types:")
    for relation, count in relation_counts.most_common():
        print(f"  {relation}: {count} connections")

Loading 1 papers...
Successfully loaded: pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf
Creating nodes using gemini_create_nodes method (25 nodes per paper)...

Extracted 25 nodes using 'gemini_create_nodes' from 'pdfs/biology/Automated Cell Structure Extraction for 3D Electron.pdf':
 1. Automated Cell Structure Extraction
 2. 3D Electron Microscopy
 3. Deep Learning for Image Segmentation
 4. U-Net Architecture
 5. Focused Ion Beam Scanning Electron Microscopy (FIB-SEM)
 6. Segment Anything Model (SAM)
 7. 3D Watershed Algorithm
 8. Fully Automated 3D Cell Model Creation
 9. Sequential Cross-Sectional Images (SBF data)
10. Segmentation of Complex Cell/Organelle Structures
11. *Cyanidioschyzon merolae* Model Organism
12. High-Resolution Intracellular Structure Information
13. Challenges of Laborious Manual Segmentation
14. Organelle-Specific Segmentation
15. Cross-Validation Methodology
16. Intersection over Union (IoU) Score
17. Multi-Directional Inference for Ac